In [3]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "..")
from pathlib import Path
import numpy as np, pandas as pd
from src.cleaning import clean_for_seq2seq

DATA = Path("../data/processed")
tix = pd.read_parquet(DATA / "tickets_sample.parquet")
tix["reply_tweet_id"] = tix["reply_tweet_id"].astype("int64")

raw = pd.read_csv("../data/raw/twcs.csv", usecols=["tweet_id", "text"],
                  dtype={"tweet_id": "int64", "text": "string"})
pairs = tix.merge(raw.rename(columns={"tweet_id": "reply_tweet_id", "text": "reply_text"}),
                  on="reply_tweet_id", how="inner")
print("Pairs found:", len(pairs))

pairs["src"] = ("co_" + pairs["company"].str.lower() + " "
                + pairs["text"].astype(str).map(clean_for_seq2seq))
pairs["tgt"] = pairs["reply_text"].astype(str).map(clean_for_seq2seq)

gold_ids = set(pd.read_csv("../data/gold/gold_to_label.csv", usecols=["tweet_id"])["tweet_id"])
pairs["is_gold"] = pairs["tweet_id"].isin(gold_ids)

for col in ["src", "tgt"]:
    n = pairs[col].str.split().str.len()
    print(f"{col} tokens: median {n.median():.0f} | 90% {n.quantile(0.9):.0f} | "
          f"95% {n.quantile(0.95):.0f} | max {n.max()}")

print("\nMost repeated replies (template check):")
print(pairs["tgt"].value_counts().head(8))

for s, t in pairs[["src", "tgt"]].sample(4, random_state=11).values:
    print(f"\nQ: {s}\nA: {t}")

pairs[["tweet_id", "company", "created_at", "src", "tgt", "is_gold"]].to_parquet(
    DATA / "ps3_pairs.parquet", index=False)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Pairs found: 100000
src tokens: median 23 | 90% 36 | 95% 45 | max 71
tgt tokens: median 22 | 90% 32 | 95% 39 | max 68

Most repeated replies (template check):
tgt
here is what you can do to work around the issue until it is fixed in a future software update                     663
here to help . send us a note at so our team can connect .                                                         258
                                                                                                                   206
to be immediately connected with our critical response line , click the call me button and enter your number at    118
here to help ! send us a note here , and our team will follow up .                                                 105
sorry for the trouble ! send us a note here , and our team will be in touch .                                       94
we are sorry to hear this was your 

In [4]:
from sklearn.model_selection import train_test_split
from src.cleaning import mask_greeting_name

pairs["tgt"] = pairs["tgt"].map(mask_greeting_name)
pairs = pairs[pairs["tgt"].str.split().str.len() >= 3].copy()
print("Pairs after dropping empty/very short replies:", len(pairs))
print("Share of replies with masked name:", round(pairs["tgt"].str.contains(r"\bxname\b").mean(), 3))

# Gold tickets are held out entirely (used later for qualitative review in the demo)
pool = pairs[~pairs["is_gold"]]
train, temp = train_test_split(pool, test_size=0.06, random_state=42)
val, test = train_test_split(temp, test_size=0.5, random_state=42)

out = pd.concat([train.assign(split="train"), val.assign(split="val"), test.assign(split="test"),
                 pairs[pairs["is_gold"]].assign(split="gold")])
out[["tweet_id", "company", "created_at", "src", "tgt", "split"]].to_parquet(DATA / "ps3_pairs.parquet", index=False)
print(out["split"].value_counts())

print("\nMasked examples:")
for t in pairs.loc[pairs["tgt"].str.contains(r"\bxname\b"), "tgt"].sample(4, random_state=2):
    print(" •", t)

Pairs after dropping empty/very short replies: 99548
Share of replies with masked name: 0.126
split
train    93201
test      2975
val       2974
gold       398
Name: count, dtype: int64

Masked examples:
 • hello xname . j . , i do apologize for the issues you are experiencing with our service and i will be happy to assist . ca .
 • hey xname ! let us check your eligibility and get you the steps to get that free netflix ! dm me here jasonburr
 • hi xname . we would be more than happy to review your account . could you please send us a dm with the address associated with your services we are better able to assist . topher
 • hi xname , please close all cc apps from activity monitor and restart the premiere pro . let us know , if this helps . if not , may i know what error are you getting ? thanks !


In [5]:
import json, time
from tensorflow import keras
from tensorflow.keras import layers
from src.experiments import log_experiment

keras.utils.set_random_seed(42)
MODELS = Path("../models")
SRC_LEN, TGT_LEN = 45, 40
SRC_VOCAB, TGT_VOCAB = 15_000, 10_000
EMB, UNITS = 128, 256

d = pd.read_parquet(DATA / "ps3_pairs.parquet")
tr, va, te = (d[d["split"] == s] for s in ("train", "val", "test"))
arr = lambda s: np.array(s.tolist(), dtype=object)
tag = lambda s: "startseq " + s + " endseq"

src_vec = layers.TextVectorization(SRC_VOCAB, output_sequence_length=SRC_LEN, standardize=None)
tgt_vec = layers.TextVectorization(TGT_VOCAB, output_sequence_length=TGT_LEN + 1, standardize=None)
src_vec.adapt(arr(tr["src"]))
tgt_vec.adapt(arr(tag(tr["tgt"])))

def encode(df):
    s = src_vec(arr(df["src"])).numpy()
    t = tgt_vec(arr(tag(df["tgt"]))).numpy()
    return {"src": s, "tgt_in": t[:, :-1]}, t[:, 1:]          # decoder input / shifted target

Xtr, ytr = encode(tr)
Xva, yva = encode(va)
mask_w = lambda y: (y != 0).astype("float32")                  # ignore padding in loss

enc_in = keras.Input((SRC_LEN,), dtype="int64", name="src")
dec_in = keras.Input((TGT_LEN,), dtype="int64", name="tgt_in")
x = layers.Embedding(SRC_VOCAB, EMB, mask_zero=True)(enc_in)
_, h, c = layers.LSTM(UNITS, return_state=True)(x)
y = layers.Embedding(TGT_VOCAB, EMB, mask_zero=True)(dec_in)
y = layers.LSTM(UNITS, return_sequences=True)(y, initial_state=[h, c])
out = layers.Dense(TGT_VOCAB, activation="softmax")(y)
s2s = keras.Model([enc_in, dec_in], out, name="seq2seq_lstm")

s2s.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy",
            weighted_metrics=["accuracy"])
t0 = time.time()
hist = s2s.fit(Xtr, ytr, sample_weight=mask_w(ytr), validation_data=(Xva, yva, mask_w(yva)),
               epochs=15, batch_size=128, verbose=2,
               callbacks=[keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)])

best_loss = min(hist.history["val_loss"])
print(f"\nBest val loss {best_loss:.3f} → perplexity {np.exp(best_loss):.1f} | {(time.time() - t0) / 60:.0f} min")
log_experiment("PS-3", "Seq2Seq LSTM (no attention)",
               {"emb": EMB, "units": UNITS, "src_len": SRC_LEN, "tgt_len": TGT_LEN},
               {"val_loss": best_loss, "perplexity": np.exp(best_loss)})

s2s.save(MODELS / "ps3_seq2seq_lstm.keras")
(MODELS / "ps3_src_vocab.json").write_text(json.dumps(src_vec.get_vocabulary()))
(MODELS / "ps3_tgt_vocab.json").write_text(json.dumps(tgt_vec.get_vocabulary()))

Epoch 1/15
729/729 - 315s - 433ms/step - accuracy: 0.1678 - loss: 4.5975 - val_accuracy: 0.2987 - val_loss: 3.6133
Epoch 2/15
729/729 - 351s - 482ms/step - accuracy: 0.3407 - loss: 3.3318 - val_accuracy: 0.3704 - val_loss: 3.1126
Epoch 3/15
729/729 - 360s - 494ms/step - accuracy: 0.3896 - loss: 2.9806 - val_accuracy: 0.4020 - val_loss: 2.8806
Epoch 4/15
729/729 - 368s - 505ms/step - accuracy: 0.4182 - loss: 2.7757 - val_accuracy: 0.4256 - val_loss: 2.7305
Epoch 5/15
729/729 - 370s - 508ms/step - accuracy: 0.4387 - loss: 2.6312 - val_accuracy: 0.4415 - val_loss: 2.6288
Epoch 6/15
729/729 - 402s - 552ms/step - accuracy: 0.4542 - loss: 2.5221 - val_accuracy: 0.4531 - val_loss: 2.5554
Epoch 7/15
729/729 - 384s - 526ms/step - accuracy: 0.4671 - loss: 2.4330 - val_accuracy: 0.4621 - val_loss: 2.4986
Epoch 8/15
729/729 - 373s - 511ms/step - accuracy: 0.4782 - loss: 2.3567 - val_accuracy: 0.4692 - val_loss: 2.4543
Epoch 9/15
729/729 - 372s - 510ms/step - accuracy: 0.4883 - loss: 2.2897 - val_a

105021

In [6]:
from src.seq2seq import greedy_decode, evaluate_generation

tgt_vocab = tgt_vec.get_vocabulary()
ev = te.sample(1000, random_state=42)
Xev, _ = encode(ev)

t0 = time.time()
hyps = greedy_decode(s2s, Xev["src"], tgt_vocab, max_len=TGT_LEN)
m = evaluate_generation(ev["tgt"].tolist(), hyps)
print({k: round(v, 4) for k, v in m.items()}, f"| {time.time() - t0:.0f}s")
log_experiment("PS-3", "Seq2Seq LSTM (no attention)", {"decode": "greedy", "n_eval": 1000}, m)

ev = ev.assign(generated=hyps)
ev[["company", "src", "tgt", "generated"]].to_csv("../reports/ps3_lstm_samples.csv", index=False)

print("\nMost frequent generated replies:\n", pd.Series(hyps).value_counts().head(5))
for s, r, g in ev[["src", "tgt", "generated"]].head(6).values:
    print(f"\nQ:   {s}\nREF: {r}\nGEN: {g}")

{'bleu': 0.0852, 'rouge1_f': 0.2655, 'rougeL_f': 0.2288, 'distinct_ratio': 0.717} | 69s

Most frequent generated replies:
 we are here to help ! send us a note via so our team can assist .                                    18
i apologize for the inconvenience . i will be happy to help . can you dm me your account number ?    17
we would like to help . dm us and we will go from there .                                            15
we are here to help . dm us and we will go from there .                                              12
here is what you can do to work around the issue until it is fixed in a future software update       12
Name: count, dtype: int64

Q:   co_amazonhelp how do i make it so , when i am looking at items in my cart , i can see the total amount i am paying for each set of items as opposed to just seeing the price for a single unit ? so if i am buying 13 units at 6 . 95 each and 3 for 7 , i can see the total price for each set ?
REF: hi , any chance you can provid

In [7]:
keras.utils.set_random_seed(42)

enc_in = keras.Input((SRC_LEN,), dtype="int64", name="src")
dec_in = keras.Input((TGT_LEN,), dtype="int64", name="tgt_in")
x = layers.Embedding(SRC_VOCAB, EMB, mask_zero=True)(enc_in)
enc_seq, h, c = layers.LSTM(UNITS, return_sequences=True, return_state=True)(x)
y = layers.Embedding(TGT_VOCAB, EMB, mask_zero=True)(dec_in)
dec_seq = layers.LSTM(UNITS, return_sequences=True)(y, initial_state=[h, c])
context = layers.Attention()([dec_seq, enc_seq])             # each decoder step attends over the input words
y = layers.Dense(UNITS, activation="tanh")(layers.Concatenate()([dec_seq, context]))
out = layers.Dense(TGT_VOCAB, activation="softmax")(y)
s2s_attn = keras.Model([enc_in, dec_in], out, name="seq2seq_lstm_attention")

s2s_attn.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy",
                 weighted_metrics=["accuracy"])
t0 = time.time()
hist_a = s2s_attn.fit(
    Xtr, ytr, sample_weight=mask_w(ytr), validation_data=(Xva, yva, mask_w(yva)),
    epochs=15, batch_size=128, verbose=2,
    callbacks=[keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
               keras.callbacks.ModelCheckpoint(MODELS / "ps3_seq2seq_attn.keras", save_best_only=True)])

best_a = min(hist_a.history["val_loss"])
print(f"\nAttention: best val loss {best_a:.3f} → perplexity {np.exp(best_a):.1f} | {(time.time() - t0) / 60:.0f} min")

hyps_a = greedy_decode(s2s_attn, Xev["src"], tgt_vocab, max_len=TGT_LEN)
m_a = evaluate_generation(ev["tgt"].tolist(), hyps_a)
log_experiment("PS-3", "Seq2Seq LSTM + attention",
               {"emb": EMB, "units": UNITS, "decode": "greedy", "n_eval": 1000},
               {**m_a, "val_loss": best_a, "perplexity": np.exp(best_a)})

print("\n               BLEU    ROUGE-1  ROUGE-L  distinct")
for name, mm in [("No attention", m), ("Attention", m_a)]:
    print(f"{name:13} {mm['bleu']:.4f}   {mm['rouge1_f']:.4f}   {mm['rougeL_f']:.4f}   {mm['distinct_ratio']:.3f}")

ev = ev.assign(generated_attn=hyps_a)
ev[["company", "src", "tgt", "generated", "generated_attn"]].to_csv("../reports/ps3_samples.csv", index=False)
for s, r, g, ga in ev[["src", "tgt", "generated", "generated_attn"]].head(6).values:
    print(f"\nQ:    {s}\nREF:  {r}\nBASE: {g}\nATTN: {ga}")

Epoch 1/15
729/729 - 315s - 433ms/step - accuracy: 0.0823 - loss: 5.2343 - val_accuracy: 0.1382 - val_loss: 4.7683
Epoch 2/15
729/729 - 356s - 489ms/step - accuracy: 0.2357 - loss: 4.1967 - val_accuracy: 0.3096 - val_loss: 3.6512
Epoch 3/15
729/729 - 345s - 473ms/step - accuracy: 0.3462 - loss: 3.3893 - val_accuracy: 0.3829 - val_loss: 3.1162
Epoch 4/15
729/729 - 354s - 485ms/step - accuracy: 0.4128 - loss: 2.8991 - val_accuracy: 0.4327 - val_loss: 2.7378
Epoch 5/15
729/729 - 326s - 448ms/step - accuracy: 0.4524 - loss: 2.5915 - val_accuracy: 0.4586 - val_loss: 2.5407
Epoch 6/15
729/729 - 310s - 426ms/step - accuracy: 0.4754 - loss: 2.4116 - val_accuracy: 0.4733 - val_loss: 2.4301
Epoch 7/15
729/729 - 322s - 442ms/step - accuracy: 0.4916 - loss: 2.2877 - val_accuracy: 0.4838 - val_loss: 2.3603
Epoch 8/15
729/729 - 331s - 454ms/step - accuracy: 0.5047 - loss: 2.1927 - val_accuracy: 0.4909 - val_loss: 2.3161
Epoch 9/15
729/729 - 339s - 465ms/step - accuracy: 0.5161 - loss: 2.1144 - val_a

In [8]:
hyps_nr = greedy_decode(s2s_attn, Xev["src"], tgt_vocab, max_len=TGT_LEN, no_repeat_ngram=3)
m_nr = evaluate_generation(ev["tgt"].tolist(), hyps_nr)
log_experiment("PS-3", "Seq2Seq LSTM + attention", {"decode": "greedy+no_repeat_3gram", "n_eval": 1000}, m_nr)

print("                     BLEU    ROUGE-1  ROUGE-L  distinct")
for name, mm in [("No attention", m), ("Attention", m_a), ("Attn + no-repeat", m_nr)]:
    print(f"{name:18} {mm['bleu']:.4f}   {mm['rouge1_f']:.4f}   {mm['rougeL_f']:.4f}   {mm['distinct_ratio']:.3f}")

ev = ev.assign(generated_attn_nr=hyps_nr)
ev.to_csv("../reports/ps3_samples.csv", index=False)
for s, ga, gn in ev[["src", "generated_attn", "generated_attn_nr"]].iloc[[2, 4, 5]].values:
    print(f"\nQ:         {s}\nATTN:      {ga}\nNO-REPEAT: {gn}")

                     BLEU    ROUGE-1  ROUGE-L  distinct
No attention       0.0852   0.2655   0.2288   0.717
Attention          0.1020   0.2827   0.2431   0.786
Attn + no-repeat   0.1020   0.2837   0.2436   0.787

Q:         co_southwestair , 4hr delay , at least 1 hr to go ? we are taken off plane to wait in line for privilege of more waiting ? flight1031
ATTN:      we are so sorry for the wait , we are so sorry for the wait . we will have you on your way as soon as possible .
NO-REPEAT: we are so sorry for the wait , we are working hard to get you on your way as soon as possible .

Q:         co_asktarget could you please make market pantry strawberry fruit snacks bigger pouches ? like the kind at checkout ? these little ones suck
ATTN:      we are so sorry about that . we have a great idea at this time . we will share your comments with our real estate team . thanks for letting us know .
NO-REPEAT: we are so sorry about that . we have a great idea at this time . we will share your co